# Inspect local training storage

Run this notebook on the OCI AI Data Platform training node before configuring training checkpoints. It reports mount points, available capacity, filesystem type, and write access for candidate directories. It creates and removes only a small temporary file in each writable candidate directory.

Do not use the Object Storage-mounted volume for transient checkpoints. Select a candidate only after confirming from the report that it is a local filesystem with sufficient free space for the full training run.

In [ ]:
from pathlib import Path

# Edit this list only when the workspace exposes additional candidate local paths.
CANDIDATE_DIRECTORIES = (
    Path('/tmp'),
    Path('/var/tmp'),
    Path('/Volumes/fine_tuning/fine_tuning/vol_finetuning'),
)
MINIMUM_AVAILABLE_GIB = 20

print(f'Minimum free-space threshold: {MINIMUM_AVAILABLE_GIB} GiB')
for candidate in CANDIDATE_DIRECTORIES:
    print(f'- {candidate}')

In [ ]:
import subprocess

result = subprocess.run(['df', '-hPT'], check=False, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'Unable to inspect mounted filesystems: {result.stderr.strip()}')

print(result.stdout)

In [ ]:
import os
import tempfile


def gibibytes(byte_count: int) -> float:
    """Convert a byte count to gibibytes."""
    return byte_count / 1024 ** 3


def inspect_candidate(directory: Path) -> dict[str, object]:
    """Inspect capacity and write access for one existing directory."""
    report = {'path': str(directory), 'exists': directory.exists(), 'is_directory': directory.is_dir()}
    if not report['is_directory']:
        return report

    resolved_directory = directory.resolve()
    filesystem_stats = os.statvfs(resolved_directory)
    available_bytes = filesystem_stats.f_bavail * filesystem_stats.f_frsize
    report.update(
        resolved_path=str(resolved_directory),
        device_id=os.stat(resolved_directory).st_dev,
        available_gib=gibibytes(available_bytes),
        meets_capacity_threshold=available_bytes >= MINIMUM_AVAILABLE_GIB * 1024 ** 3,
    )

    try:
        with tempfile.NamedTemporaryFile(
            mode='wb', dir=resolved_directory, prefix='.storage-check-', delete=True
        ) as temporary_file:
            temporary_file.write(b'OCI AI Data Platform storage check\n')
            temporary_file.flush()
            os.fsync(temporary_file.fileno())
        report['write_check'] = 'passed'
    except OSError as error:
        report['write_check'] = f'failed: {error}'

    filesystem_result = subprocess.run(
        ['df', '-hPT', str(resolved_directory)], check=False, capture_output=True, text=True
    )
    report['filesystem'] = (
        filesystem_result.stdout.strip()
        if filesystem_result.returncode == 0
        else f'Unable to identify filesystem: {filesystem_result.stderr.strip()}'
    )
    return report


for candidate in CANDIDATE_DIRECTORIES:
    report = inspect_candidate(candidate)
    print(f"\nCandidate: {report['path']}")
    for key, value in report.items():
        if key != 'path':
            print(f'  {key}: {value}')

## Decision checklist

Choose a directory only when its report shows `write_check: passed`, `meets_capacity_threshold: True`, and a local filesystem type or mount point. Record the chosen path and its available capacity before changing the training notebook. The Object Storage volume is for the final adapter copy only.